In [22]:
'''
WHY DATA LOADER?

X_train_tensor = 1003 rows

10000 epoch
1 epoch --> 1003 rows as input 

lets say we have 1M rows in X_train_tensor
so in 1 epoch we sent all 1M rows hence more ram usage

lets say ram used is 10gb and system has 8gb ram

hence we will face out of memory error 


We are teaching human and we have a book of 1000 pages 
and after learning 1000 pages he said i m having issue in understanding,
lets say student have issue on 5th page but he can say 
that only after he have read 1000 pages,
hence we tell that student that if you read 10 pages than
tell us whether you understand or not, this works as a feedback
and same for NN will can update the weights and bias more faster and efficiently

1000 pages
10 at a time
100 feedback 
so on 100 feedback our 1 epoch will be completed

100 feedback = 1000 pages = 1 epoch

'''

'\nWHY DATA LOADER?\n\nX_train_tensor = 1003 rows\n\n10000 epoch\n1 epoch --> 1003 rows as input \n\nlets say we have 1M rows in X_train_tensor\nso in 1 epoch we sent all 1M rows hence more ram usage\n\nlets say ram used is 10gb and system has 8gb ram\n\nhence we will face out of memory error \n\n\nWe are teaching human and we have a book of 1000 pages \nand after learning 1000 pages he said i m having issue in understanding,\nlets say student have issue on 5th page but he can say \nthat only after he have read 1000 pages,\nhence we tell that student that if you read 10 pages than\ntell us whether you understand or not, this works as a feedback\nand same for NN will can update the weights and bias more faster and efficiently\n\n1000 pages\n10 at a time\n100 feedback \nso on 100 feedback our 1 epoch will be completed\n\n100 feedback = 1000 pages = 1 epoch\n\n'

# Components of Data Loader

1. Dataset (torch.utils.data.Dataset)
2. Data Loader (torch.utils.data.DataLoader) --> Wraps the dataset in iterable object with some batch.
   

In [23]:
## Creating our Custom Dataset in pytorch

# init() -> initialised the dataset, loads data, applied preprocessing
# len() -> returns total number of samples in the dataset
# getitem() -> Defines how to retrieve a single data sample when an index is provided


In [24]:
import torch 
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim 
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [25]:
df = pd.read_csv('/kaggle/input/datasets/mragpavank/insurance1/insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [26]:
df['smoker'] = df['smoker'].map({'yes' : 1 , 'no' : 0})

In [27]:
ohe=OneHotEncoder()
scaler = StandardScaler()

In [28]:
X = df.drop('charges',axis=1)
y=df['charges']

In [29]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.25,random_state=42)

In [30]:
categorical_columns = X.select_dtypes(include='object').columns
numeric_columns = X.select_dtypes(exclude='object').columns

In [31]:
preprocessor = ColumnTransformer(
                [("OneHotEncoder",ohe,categorical_columns),
                ("Standard Scaler",scaler,numeric_columns)],remainder='passthrough'
                )

In [32]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [33]:
class LinearRegression(nn.Module):
    def __init__(self,input_dim):
        super(LinearRegression,self).__init__()
        
        self.network = nn.Sequential(
                nn.Linear(input_dim,64),
                nn.ReLU(),
                nn.Linear(64,128),
                nn.ReLU(),
                nn.Linear(128,1)
        )

    def forward(self,x):
        return self.network(x)
        

In [34]:
X_train.shape

(1003, 10)

In [35]:
input_dim = X_train.shape[1]
model = LinearRegression(input_dim)

In [36]:

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(),lr = 0.01)

In [37]:
print(model)

LinearRegression(
  (network): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [38]:
class InsuranceDataset(Dataset):
    def __init__(self,X,y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        features = torch.tensor(self.X[idx],dtype =torch.float32)
        target = torch.tensor(self.y.values[idx],dtype =torch.float32)
        return features,target


In [39]:
dataset = InsuranceDataset(X_train,y_train)


In [40]:
dataloader = DataLoader(dataset,batch_size = 32,shuffle = True,num_workers = 4)

# num_workers --> When num_workers=0, data loading occurs in the main process, 
# which can slow down training as the model waits for data to be fetched. 
# Setting num_workers > 0 enables parallel data loading using multiple subprocesses. 
# Each worker independently loads batches of data and sends them to the main process, 
# allowing the GPU to process data while the next batch is being prepared.

In [41]:
for batch_idx,(features,targets) in enumerate(dataloader):
    print(f"Batch {batch_idx + 1}")
    print("Features : ",features.shape)
    print("Targets : ",targets.shape)
    break

Batch 1
Features :  torch.Size([32, 10])
Targets :  torch.Size([32])


1000 --> total number of data (say)

- epoch = 100
- batch_size = 1
- 1000/1 = 1000

1000 iteration in each epoch

100 * 1000 = total_iterations

In [ ]:
## Training Loop
epochs = 1000
for epoch in range(epochs):
    model.train()

    for batch_idx,(batch_X,batch_y) in enumerate(dataloader):
        print(f"Current batch: {batch_idx}")
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions,batch_y.view(-1,1))
        loss.backward()
        optimizer.step()
        
        print(f"Batch[{batch_idx+1}/{epochs}], Loss : {loss.item():.4f}")


Current batch: 0
Batch[1/1000], Loss : 355355936.0000
Current batch: 1
Batch[2/1000], Loss : 359135584.0000
Current batch: 2
Batch[3/1000], Loss : 253641504.0000
Current batch: 3
Batch[4/1000], Loss : 418682784.0000
Current batch: 4
Batch[5/1000], Loss : 326348224.0000
Current batch: 5
Batch[6/1000], Loss : 625897920.0000
Current batch: 6
Batch[7/1000], Loss : 355319584.0000
Current batch: 7
Batch[8/1000], Loss : 487740032.0000
Current batch: 8
Batch[9/1000], Loss : 324289280.0000
Current batch: 9
Batch[10/1000], Loss : 261340800.0000
Current batch: 10
Batch[11/1000], Loss : 324934112.0000
Current batch: 11
Batch[12/1000], Loss : 152503904.0000
Current batch: 12
Batch[13/1000], Loss : 183376544.0000
Current batch: 13
Batch[14/1000], Loss : 310294336.0000
Current batch: 14
Batch[15/1000], Loss : 206641664.0000
Current batch: 15
Batch[16/1000], Loss : 235839232.0000
Current batch: 16
Batch[17/1000], Loss : 126041992.0000
Current batch: 17
Batch[18/1000], Loss : 313232224.0000
Current bat